In [3]:
import os
import shutil
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

### 复制需要分析的中心线（平滑化后的MCA和ICA --> _ICA_MCA_spline.ply ）

In [5]:
# 源目录（包含你的ICA_MCA_ACA_centerline文件夹）
src_dir = r"D:\github\PhD\nagahma\data\04_ICA_MCA_ACA_centerline"

# 目标目录
dst_dir = r"D:\github\PhD\nagahma\data\feng\MCA_ICA"

# 如果目标目录不存在则创建
os.makedirs(dst_dir, exist_ok=True)

# 遍历源目录中的所有文件
for file_name in os.listdir(src_dir):
    # 只匹配后缀为 _ICA_MCA_spline.ply 的文件
    if file_name.endswith("_ICA_MCA_spline.ply"):
        src_path = os.path.join(src_dir, file_name)
        dst_path = os.path.join(dst_dir, file_name)
        shutil.copy2(src_path, dst_path)  # 保留原文件时间戳
        #print(f"已复制: {file_name}")

print("All done.")

All done.


### 抽取出ICA部分并转换成vtk格式(--> _ICA_spline.vtk)

In [8]:
# === 输入输出路径 ===
input_dir = r"D:\github\PhD\nagahma\data\feng\MCA_ICA"
output_dir = r"D:\github\PhD\nagahma\data\feng\ICA(vtk)"
os.makedirs(output_dir, exist_ok=True)

# === 固定颜色 ===
COLOR_L = ('80', '160', '160')  # L_ICA
COLOR_R = ('0', '208', '192')   # R_ICA

def parse_vertices(ply_path):
    with open(ply_path, 'r') as f:
        lines = f.readlines()
    end_header = None
    nverts = 0
    for i, l in enumerate(lines):
        if l.startswith('element vertex'):
            nverts = int(l.split()[-1])
        if l.strip() == 'end_header':
            end_header = i
            break
    verts = []
    for l in lines[end_header + 1:end_header + 1 + nverts]:
        parts = l.strip().split()
        if len(parts) >= 6:
            x, y, z = map(float, parts[:3])
            r, g, b = parts[3:6]
            verts.append((x, y, z, r, g, b))
        elif len(parts) == 3:
            x, y, z = map(float, parts)
            verts.append((x, y, z, None, None, None))
    return verts

def write_vtk(points_xyz, out_path, polyline=True):
    with open(out_path, 'w') as f:
        f.write("# vtk DataFile Version 3.0\n")
        f.write(os.path.basename(out_path) + "\n")
        f.write("ASCII\n")
        f.write("DATASET POLYDATA\n")
        f.write(f"POINTS {len(points_xyz)} float\n")
        for x, y, z in points_xyz:
            f.write(f"{x} {y} {z}\n")
        if polyline and len(points_xyz) > 1:
            f.write(f"LINES 1 {len(points_xyz)+1}\n")
            f.write(f"{len(points_xyz)} " + " ".join(str(i) for i in range(len(points_xyz))) + "\n")

for fn in os.listdir(input_dir):
    if not fn.endswith("_ICA_MCA_spline.ply"):
        continue
    src = os.path.join(input_dir, fn)

    # 判断L还是R
    if "_L_ICA_" in fn:
        target_color = COLOR_L
    elif "_R_ICA_" in fn:
        target_color = COLOR_R
    else:
        print(f"[WARN] 文件名中未找到L/R: {fn}")
        continue

    verts = parse_vertices(src)

    # 抽取对应颜色
    ica_pts = [(x, y, z) for x, y, z, r, g, b in verts
               if r == target_color[0] and g == target_color[1] and b == target_color[2]]

    if not ica_pts:
        print(f"[WARN] {fn} 没找到指定颜色 {target_color}")
        continue

    # 输出文件名改成 _ICA_spline.vtk
    out_name = fn.replace("_ICA_MCA_spline.ply", "_ICA_spline.vtk")
    dst_vtk = os.path.join(output_dir, out_name)

    write_vtk(ica_pts, dst_vtk, polyline=True)
    #print(f"[OK] {fn} → {dst_vtk} (点数={len(ica_pts)})")

print("All done.")


All done.


In [10]:
import os

input_dir = r"D:\github\PhD\nagahma\data\feng\ICA(vtk)"
output_dir = r"D:\github\PhD\nagahma\data\feng\ICA(txt)"  # 输出目录
os.makedirs(output_dir, exist_ok=True)

for fn in os.listdir(input_dir):
    if not fn.endswith("_ICA_spline.vtk"):
        continue
    vtk_path = os.path.join(input_dir, fn)
    txt_path = os.path.join(output_dir, os.path.splitext(fn)[0] + ".txt")

    with open(vtk_path, 'r') as f:
        lines = f.readlines()

    # 找到 POINTS 段
    points_start = None
    n_points = 0
    for i, l in enumerate(lines):
        if l.strip().startswith("POINTS"):
            parts = l.strip().split()
            if len(parts) >= 2:
                n_points = int(parts[1])
            points_start = i + 1
            break

    if points_start is None or n_points == 0:
        print(f"[WARN] 未找到POINTS段: {fn}")
        continue

    # 读取坐标
    coords = []
    for l in lines[points_start:]:
        parts = l.strip().split()
        if len(parts) < 3:
            continue
        x, y, z = map(float, parts[:3])
        coords.append((x, y, z))
        if len(coords) >= n_points:
            break

    # === 在头部写点数 ===
    header_line = f"#spline3d {len(coords)}\n"

    # 写出 txt 文件
    with open(txt_path, 'w') as out:
        out.write(header_line)
        for x, y, z in coords:
            out.write(f"{x} {y} {z}\n")

    #print(f"[OK] {fn} → {txt_path} (点数={len(coords)})")

print("All done.")


All done.


In [ ]:
# === 读取 PLY 文件 ===
file_path = r"D:\github\PhD\nagahma\data\feng\00000000.nii_L_ICA_MCA_spline.ply"

vertices = []
colors = []

with open(file_path, 'r') as f:
    lines = f.readlines()

# 找到end_header的位置
end_header_idx = None
for i, line in enumerate(lines):
    if line.strip() == "end_header":
        end_header_idx = i
        break

# 顶点数量
num_vertices = 0
for line in lines[:end_header_idx+1]:
    if line.startswith("element vertex"):
        num_vertices = int(line.split()[-1])

# 提取顶点数据
vertex_lines = lines[end_header_idx+1:end_header_idx+1+num_vertices]
for line in vertex_lines:
    parts = line.strip().split()
    if len(parts) >= 6:
        x, y, z = map(float, parts[:3])
        r, g, b = map(int, parts[3:6])
        vertices.append((x, y, z))
        colors.append((r/255, g/255, b/255))  # matplotlib颜色范围0-1

vertices = pd.DataFrame(vertices, columns=['x', 'y', 'z'])

# === 绘制 3D 散点图，1:1 比例 ===
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(vertices['x'], vertices['y'], vertices['z'], c=colors, s=10)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('ICA + MCA Centerline (1:1 scale)')

# 设置轴范围相同，保持1:1比例
xlim = (vertices['x'].min(), vertices['x'].max())
ylim = (vertices['y'].min(), vertices['y'].max())
zlim = (vertices['z'].min(), vertices['z'].max())

max_range = max(xlim[1]-xlim[0], ylim[1]-ylim[0], zlim[1]-zlim[0]) / 2.0
mid_x = (xlim[1]+xlim[0]) * 0.5
mid_y = (ylim[1]+ylim[0]) * 0.5
mid_z = (zlim[1]+zlim[0]) * 0.5

ax.set_xlim(mid_x - max_range, mid_x + max_range)
ax.set_ylim(mid_y - max_range, mid_y + max_range)
ax.set_zlim(mid_z - max_range, mid_z + max_range)

plt.tight_layout()
plt.show()
